# 워크숍 3 — 탐색: `go_to` 다시 구현하기

**예상 소요 시간:** 약 90분  
**선수 학습:** 워크숍 1, 2

내장 `go_to` 스킬은 내부가 보이지 않는 블랙박스입니다. 이번에는 두 방식으로 직접 구현합니다.

- **파트 A — 전역 상태 탐색:** `scene_state`와 `robot_status`의 자세 데이터로 방향 오차를 계산하고 비례 제어기로 주행합니다.
- **파트 B — 비전 전용 탐색:** 특별한 위치 데이터 없이 카메라의 색상 검출과 블롭 면적만으로 탐색합니다.

GPS나 위치 추정을 사용할 수 없는 실제 로봇 환경을 이해하려면 두 접근법을 모두 아는 것이 중요합니다.

> ⚠️ **시작하기 전에:** 이전 워크숍의 시뮬레이터 창을 모두 닫으세요. 노트북마다 새 로봇을 만들므로 뷰어 탭을 두 개 동시에 열면 문제가 생깁니다.

## 섹션 1 — 빠른 설정

In [ ]:
%pip install -q "menlo-robot-sdk[livekit]" python-dotenv opencv-python matplotlib

In [ ]:
# 워크숍 1과 동일한 설정 — 자세한 설명은 해당 노트북을 참고하세요
import asyncio
import math
import os
import time

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image

from menlo_robot_sdk import AsyncClient, connect
from menlo_robot_sdk.experimental import generate_room_key

# ── 키 로더: 먼저 .env를 확인한 다음 Colab Secrets를 확인합니다 ────────────────────────
def _load_keys():
    # 모드 B: 로컬 .env 파일
    try:
        from dotenv import load_dotenv
        load_dotenv(override=False)
        if os.environ.get("MENLO_API_KEY"):
            print("Loaded keys from .env file")
            return "dotenv"
    except ImportError:
        pass  # python-dotenv not installed → not in .env mode

    # 모드 A: Google Colab Secrets
    try:
        from google.colab import userdata
        os.environ.setdefault("MENLO_API_KEY",    userdata.get("MENLO_API_KEY") or "")
        os.environ.setdefault("TOKAMAK_API_KEY",  userdata.get("TOKAMAK_API_KEY") or "")
        print("Loaded keys from Colab Secrets")
        return "colab"
    except Exception:
        pass

    return "env"  # keys already in environment (CI, Docker, etc.)

_load_keys()

MENLO_API_KEY   = os.environ.get("MENLO_API_KEY", "")
TOKAMAK_API_KEY = os.environ.get("TOKAMAK_API_KEY", "")
RCS_URL         = "https://platform-auth.menlo.ai/rcs"
VIEWER_BASE_URL = "https://sim.menlo.ai"

assert MENLO_API_KEY and not MENLO_API_KEY.startswith("sk_live_YOUR"), (
    "MENLO_API_KEY not set!\n"
    "  • Colab: add it in the Secrets manager (key icon in the left sidebar)\n"
    "  • Local: add MENLO_API_KEY=... to a .env file next to this notebook"
)
print(f"Configuration loaded — platform: {RCS_URL}")
print(f"MENLO_API_KEY    : {MENLO_API_KEY[:12]}...")

In [ ]:
client = AsyncClient(rcs_url=RCS_URL, api_key=MENLO_API_KEY)

r = await client.robots.create(name=f"workshop3-{int(time.time())}", model="asimov-v0")
robot_id = r.robot.id
await client.robots.create_session(robot_id)

room_key = await generate_room_key(client, robot_id)
print(f"VIEWER URL:\n{VIEWER_BASE_URL}/?key={room_key}")

> 뷰어 URL을 **Google Chrome**에서 열고 장면이 로드될 때까지 기다린 뒤 아래 셀을 실행하세요.

In [ ]:
session = await connect(
    client, robot_id,
    worker_names=[], rcw_identity_prefix="simplesim", join_livekit=True,
)
print(f"Connected: {session.session.room_name}")


async def wait_for_skills(timeout_s: float = 180.0):
    """뷰어가 접속하고 스킬을 사용할 수 있을 때까지 주기적으로 확인합니다."""
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        try:
            found = await session.discover_skills()
            if found:
                return found
        except (RuntimeError, TimeoutError):
            pass  # viewer not joined yet
        await asyncio.sleep(2.0)
    raise TimeoutError("Viewer did not join — is the Chrome tab open?")


skills = await wait_for_skills()
print(f"Skills: {[s.name for s in skills]}")

---
# 파트 A — 전역 상태 탐색

파트 A에서는 `scene_state`와 `robot_status`의 월드 좌표를 사용합니다. 가장 안정적이지만 시뮬레이션의 특별한 데이터에 접근해야 합니다.

## 섹션 2 — 로봇 자세와 월드 좌표

창고는 2D 좌표계를 사용합니다.
- **yaw=0°**: +x 방향
- **yaw=90°**: +y 방향
- **bearing**=`atan2(dy, dx)`: 월드 좌표에서 목표를 향한 각도
- **heading_error**: 목표를 바라보기 위해 로봇이 돌아야 할 각도

In [ ]:
# 읽기: 로봇 자세와 목표 위치
status = await session.state.get("robot_status")
scene  = await session.state.get("scene_state")

rx, ry = status.robot.pose.position[0], status.robot.pose.position[1]
yaw    = status.robot.pose.yaw_deg

pad_c  = scene.entities["pad_C"]
tx, ty = pad_c.pose.position[0], pad_c.pose.position[1]

print(f"Robot   : x={rx:.2f}, y={ry:.2f}, yaw={yaw:.1f}°")
print(f"pad_C   : x={tx:.2f}, y={ty:.2f}")

# 기하 계산
dist          = math.hypot(tx - rx, ty - ry)
bearing       = math.degrees(math.atan2(ty - ry, tx - rx))
heading_error = (bearing - yaw + 180) % 360 - 180

print(f"\nDistance      : {dist:.2f} m")
print(f"Bearing       : {bearing:.1f}° (world frame)")
print(f"Heading error : {heading_error:.1f}° (how much to turn)")

## 섹션 3 — 목표를 바라보도록 회전하기

In [ ]:
async def turn_to_face(session, target_pos):
    """Turn the robot to face target_pos = [x, y, ...]."""
    state = await session.state.get("robot_status")
    rx, ry = state.robot.pose.position[0], state.robot.pose.position[1]
    yaw = state.robot.pose.yaw_deg
    tx, ty = target_pos[0], target_pos[1]

    bearing = math.degrees(math.atan2(ty - ry, tx - rx))
    heading_error = (bearing - yaw + 180) % 360 - 180
    print(f"  Heading error: {heading_error:+.1f}°")

    if abs(heading_error) < 5:
        print("  Already facing target.")
        return

    # wz > 0 왼쪽으로 회전, wz < 0 오른쪽으로 회전
    wz = 0.4 if heading_error > 0 else -0.4
    # 지속 시간 = 각도 / 각속도(라디안 단위)
    duration = min(abs(heading_error) / (abs(wz) * 57.296), 8.0)

    await session.invoke(
        "set_velocity",
        {"vx": 0.15, "vy": 0.0, "wz": wz, "duration_s": duration},
        timeout_s=30,
    )
    print(f"  Turned {heading_error:+.0f}° in {duration:.2f}s")


# 실행 예시: pad_C를 바라보도록 회전
scene = await session.state.get("scene_state")
pad_c_pos = scene.entities["pad_C"].pose.position
print("Turning to face pad_C...")
await turn_to_face(session, pad_c_pos)

## 섹션 4 — 목표를 향해 주행하기

In [ ]:
async def drive_to_distance(session, target_pos, tolerance=0.6):
    """Drive forward until within tolerance meters of target_pos."""
    for step in range(20):
        state = await session.state.get("robot_status")
        rx, ry = state.robot.pose.position[0], state.robot.pose.position[1]
        dist = math.hypot(target_pos[0] - rx, target_pos[1] - ry)
        print(f"  Step {step+1}: dist={dist:.2f}m")
        if dist <= tolerance:
            print("  도착!")
            return True
        vx = min(0.8, dist * 0.4)  # proportional speed
        await session.invoke(
            "set_velocity",
            {"vx": vx, "vy": 0.0, "wz": 0.0, "duration_s": 0.8},
            timeout_s=30,
        )
    print("  Max iterations reached.")
    return False


# 실행 예시: pad_C를 바라본 뒤 전진
print("Driving toward pad_C...")
await drive_to_distance(session, pad_c_pos, tolerance=0.8)

## 섹션 5 — 완성된 `my_go_to_global`

회전과 주행을 반복 보정 루프로 결합합니다.

In [ ]:
async def my_go_to_global(session, entity_id, tolerance=0.6, max_iters=10):
    """Navigate to entity_id using scene_state + robot_status."""
    scene = await session.state.get("scene_state")
    entity = scene.entities.get(entity_id)
    if entity is None:
        print(f"Entity '{entity_id}' not found in scene.")
        return False
    target_pos = entity.pose.position

    for i in range(max_iters):
        state = await session.state.get("robot_status")
        rx, ry = state.robot.pose.position[0], state.robot.pose.position[1]
        dist = math.hypot(target_pos[0] - rx, target_pos[1] - ry)
        print(f"Iter {i+1}: dist={dist:.2f}m to {entity_id}")
        if dist <= tolerance:
            print(f"Reached {entity_id}!")
            return True
        await turn_to_face(session, target_pos)
        await drive_to_distance(session, target_pos, tolerance=tolerance)

    print("Max iterations reached.")
    return False


# 실행 예시: pad_C까지 이동
print("my_go_to_global → pad_C")
success = await my_go_to_global(session, "pad_C")
print(f"Result: {success}")

---
# 파트 B — 비전 전용 탐색

실제 환경에서는 정확한 자세 데이터를 사용할 수 없을 수 있습니다. 파트 B에서는 `scene_state`와 `robot_status` 없이 카메라만 사용합니다.

## 섹션 6 — 비전 전용 방식이 중요한 이유

- **시뮬레이션에서 현실로 전이:** 실제 하드웨어에는 정확한 좌표 같은 특별한 데이터가 거의 없습니다.
- **보편적인 센서:** 카메라는 가장 흔한 온보드 센서이며 전역 정보는 항상 제공되지 않습니다.
- **어려움:** 비전에는 노이즈가 있고 조명이 달라지며 물체가 가려질 수 있어 더 견고한 알고리즘이 필요합니다.

핵심은 "큐브의 월드 좌표는 어디인가?" 대신 "카메라에서 큐브가 어디에 보이며 거리를 어떻게 줄일까?"라고 묻는 것입니다.

## 섹션 7 — 독립적으로 실행 가능한 `perceive()` 함수

노트북 하나만으로 실행할 수 있도록 워크숍 2의 함수를 복사합니다.

In [ ]:
# 워크숍 2에서 복사했으며 다른 노트북을 불러올 필요가 없습니다
async def perceive(session):
    """Return {color: {angle_deg, blob_area}} for each visible cube color."""
    jpeg = await session.get_vision("pov")
    img = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_COLOR)
    h, w = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    COLOR_RANGES = {
        "red":    [(np.array([0,  50, 50]), np.array([10,  255, 255])),
                   (np.array([160,50, 50]), np.array([180, 255, 255]))],
        "green":  [(np.array([40, 50, 50]), np.array([80,  255, 255]))],
        "blue":   [(np.array([100,50, 50]), np.array([130, 255, 255]))],
        "yellow": [(np.array([20, 50, 50]), np.array([35,  255, 255]))],
    }

    result = {}
    for color, ranges in COLOR_RANGES.items():
        mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
        for lo, hi in ranges:
            mask |= cv2.inRange(hsv, lo, hi)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        largest = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest)
        if area < 200:
            continue
        M = cv2.moments(largest)
        if M["m00"] == 0:
            continue
        cx = int(M["m10"] / M["m00"])
        angle_deg = (cx - w / 2) / (w / 2) * 30.0
        result[color] = {"angle_deg": round(angle_deg, 1), "blob_area": int(area)}

    return result


# 간단한 테스트
obs = await perceive(session)
print("Current perception:")
for color, data in obs.items():
    print(f"  {color}: angle={data['angle_deg']:+.1f}°, area={data['blob_area']}px²")

## 섹션 8 — 목표 색상을 화면 중앙에 맞추기

목표 색상의 블롭이 카메라 중앙 ±10° 안에 들어올 때까지 회전합니다.

In [ ]:
async def center_on_color(session, target_color, angle_tolerance=10.0, max_steps=12):
    """Turn until target_color blob is within angle_tolerance of camera center."""
    for step in range(max_steps):
        obs = await perceive(session)
        if target_color not in obs:
            print(f"  Step {step+1}: {target_color} not visible — scanning...")
            await session.invoke(
                "set_velocity",
                {"vx": 0.15, "vy": 0.0, "wz": 0.3, "duration_s": 1.0},
                timeout_s=15,
            )
            continue

        angle = obs[target_color]["angle_deg"]
        print(f"  Step {step+1}: {target_color} at {angle:+.1f}°")

        if abs(angle) <= angle_tolerance:
            print("  Target 중앙 정렬 완료!")
            return True

        # 중앙을 향해 회전: 양수 각도면 물체가 오른쪽이므로 음수 wz로 오른쪽 회전
        wz = -0.3 if angle > 0 else 0.3
        await session.invoke(
            "set_velocity",
            {"vx": 0.15, "vy": 0.0, "wz": wz, "duration_s": 0.8},
            timeout_s=15,
        )

    print("  Max steps reached.")
    return False


print("Centering on red cube...")
success = await center_on_color(session, "red")
print(f"Centered: {success}")

## 섹션 9 — 블롭 크기로 주행하기

로봇이 물체에 가까워질수록 이미지의 블롭이 커집니다. 블롭 면적을 거리의 대용값으로 사용해 임곗값을 넘을 때까지 전진합니다.

**경험적 기준:** `ARRIVAL_AREA=8000 px²`는 대략 큐브에 손이 닿는 거리입니다.

In [ ]:
ARRIVAL_AREA = 8000  # pixels² — tune if needed

async def drive_toward_color(session, target_color, arrival_area=ARRIVAL_AREA, max_steps=15):
    """Drive forward toward target_color using blob area as proximity proxy."""
    for step in range(max_steps):
        obs = await perceive(session)
        if target_color not in obs:
            print(f"  Step {step+1}: {target_color} not visible")
            return False

        area  = obs[target_color]["blob_area"]
        angle = obs[target_color]["angle_deg"]
        print(f"  Step {step+1}: area={area}px², angle={angle:+.1f}°")

        if area >= arrival_area:
            print("  도착(블롭이 충분히 큼)!")
            return True

        # 경로 유지를 위해 두 단계마다 다시 중앙 정렬
        if step % 2 == 1 and abs(angle) > 10:
            wz = -0.3 if angle > 0 else 0.3
            await session.invoke(
                "set_velocity",
                {"vx": 0.15, "vy": 0.0, "wz": wz, "duration_s": 0.5},
                timeout_s=15,
            )
        else:
            await session.invoke(
                "set_velocity",
                {"vx": 0.5, "vy": 0.0, "wz": 0.0, "duration_s": 1.0},
                timeout_s=15,
            )

    print("  Max steps reached.")
    return False


print("Driving toward red cube...")
success = await drive_toward_color(session, "red")
print(f"Arrived: {success}")

## 섹션 10 — 완성된 `my_go_to_visual`

In [ ]:
async def my_go_to_visual(session, target_color, arrival_area=ARRIVAL_AREA):
    """Navigate to a cube of target_color using only the camera. No scene_state used."""
    print(f"Navigating to {target_color} cube (vision only)...")
    centered = await center_on_color(session, target_color)
    if not centered:
        print("Could not center on target.")
        return False
    return await drive_toward_color(session, target_color, arrival_area=arrival_area)


# 보이는 큐브 색상으로 실행 예시
obs = await perceive(session)
if obs:
    demo_color = next(iter(obs))  # pick first detected color
    print(f"Demo: navigating to {demo_color}")
    await my_go_to_visual(session, demo_color)
else:
    print("No cubes visible — move the robot closer to the conveyor belt first.")

---
## 연습 문제

### 연습 문제 1 — 위치와 거리 출력(파트 A)

로봇과 `pad_C`의 현재 `(x, y)` 위치를 읽어 직선거리를 계산하고 출력하세요.

In [ ]:
## 연습 문제 1: 로봇과 pad_C 위치 및 거리 출력
# 여기에 코드를 작성하세요

# Expected output:
# Robot: (1.23, 0.45)
# pad_C: (5.60, 2.10)
# Distance: 4.62 m

### 연습 문제 2 — pad_C를 바라보도록 회전(파트 A)

`scene_state`에서 `pad_C` 위치를 가져와 `turn_to_face(session, pad_c_pos)`를 호출하세요. 회전 전후 방향 오차를 출력하세요.

In [ ]:
## 연습 문제 2: pad_C를 바라보도록 회전
# 여기에 코드를 작성하세요

# Expected output:
# 실행 전: heading_error = +42.3°
# [회전 실행]
# 실행 후: heading_error ≈ 0–5°

### 연습 문제 3 — pad_C까지 주행(파트 A)

`pad_C`를 바라본 뒤 `drive_to_distance(session, pad_c_pos, tolerance=0.8)`을 호출하고 반복할 때마다 거리를 출력하세요.

In [ ]:
## 연습 문제 3: pad_C까지 주행
# 여기에 코드를 작성하세요

# Expected output:
# 단계 1: dist=4.62m
# 단계 2: dist=3.45m
# ...
# 도착!

### 연습 문제 4 — pad_D에서 `my_go_to_global` 시험(파트 A)

`my_go_to_global(session, "pad_D")`를 호출하고 반환된 뒤 `scene_state`에서 `pad_D`까지의 최종 거리를 출력해 도착을 검증하세요.

In [ ]:
## 연습 문제 4: pad_D까지 이동하고 검증
# 여기에 코드를 작성하세요

# 예상 결과: final distance < 0.6m
# e.g.: "Arrived at pad_D. Final distance: 0.42m"

### 연습 문제 5(도전) — `my_go_to_global`과 내장 `go_to` 경로 비교(파트 A)

두 방식으로 `pad_B`까지 이동하는 동안 1초마다 로봇 위치를 기록하고 `matplotlib`의 2D 산점도로 두 경로를 그리세요.

In [ ]:
## 연습 문제 5 (stretch): pad_B까지 경로 비교
# 여기에 코드를 작성하세요

# 힌트:
# 1. 1초마다 robot_status를 읽어 목록에 추가하는 백그라운드 작업 시작
# 2. Run my_go_to_global(session, "pad_B")
# 3. 내장 go_to로 반복
# 4. plt.plot(xs, ys) 각 경로에 시작/끝과 이름을 표시

### 연습 문제 6 — 카메라 각도 오프셋 측정(파트 B)

`perceive(session)`으로 빨간 큐브의 `angle_deg`를 출력하세요. 그런 다음 `scene_state`와 `robot_status`로 실제 기준 각도를 계산해 비교하세요.

In [ ]:
## 연습 문제 6: 카메라 각도와 scene_state 실제 기준값 비교
# 여기에 코드를 작성하세요

# Expected output:
# 카메라 각도: +14.2°
# 실제 장면 값:  +12.8°
# 오차:         1.4°

### 연습 문제 7 — 빨간 큐브를 중앙에 맞추기(파트 B)

`center_on_color(session, "red")`를 호출하고 매 단계 각도 오프셋을 출력하세요. 값이 0°에 수렴하는 모습을 관찰하세요.

In [ ]:
## 연습 문제 7: 빨간 큐브를 중앙에 맞추기
# 여기에 코드를 작성하세요

# Expected output:
# 단계 1: red at +18.5°
# 단계 2: red at +9.2°
# 단계 3: red at +3.1° — Target 중앙 정렬 완료!

### 연습 문제 8 — 빨간 큐브를 향해 주행(파트 B)

중앙 정렬 후 `drive_toward_color(session, "red")`를 호출하고 매 단계 블롭 면적을 출력하세요. 가까워질수록 면적이 커지는지 관찰하세요.

In [ ]:
## 연습 문제 8: 블롭 크기로 빨간 큐브를 향해 주행
# 여기에 코드를 작성하세요

# Expected output:
# 단계 1: area=1240px², angle=+2.1°
# 단계 2: area=2870px², angle=+1.8°
# ...
# 도착(블롭이 충분히 큼)!

### 연습 문제 9(도전) — 파란 큐브까지 비전 전용 탐색(파트 B)

`my_go_to_visual(session, "blue")`을 호출하세요. 완료 후 `scene_state`로 파란 큐브까지의 실제 거리를 확인하세요. 얼마나 가까이 갔나요?

In [ ]:
## 연습 문제 9 (stretch): 파란 큐브까지 비전 전용 탐색 후 거리 확인
# 여기에 코드를 작성하세요

# 예상 결과:
# 탐색 완료
# 파란 큐브까지 최종 거리: ~0.5–1.5m
# (비전 전용 탐색은 전역 상태 탐색보다 정확도가 낮습니다)

---
## 정리

In [ ]:
await session.disconnect()
await client.robots.delete(robot_id)
print("Robot deleted.")